# 04 · Anticipation baseline (Leg 1)

This notebook fits the first monthly logit for the probability that a non-member joins the S&P 500 within three months. It follows section 3.4 of the project plan: estimate on 2000–2012 and test out of sample, reporting AUC and top-20 precision. Run `01_inventory.ipynb`'s prerequisites first: this notebook runs the inventory itself and reuses `features`.

**Scope.** This is a size-and-liquidity baseline. It does not yet use earnings eligibility, float, MidCap 400 membership, or sector weight, because those inputs are not point-in-time ready. The test ends in September 2022 because verified membership ends in December 2022.

**What the labels mean.** `y3` is built from effective dates, so these metrics measure whether the pipeline ranks future entrants. They are not a return test, and a stock can already be announced at the formation date. The lead-time table below shows how much of the result could depend on that.

In [ ]:
from pathlib import Path

from IPython.utils.capture import capture_output
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

save_prepared = True
top_n = 20
bootstrap_draws = 2000
bootstrap_block_months = 12
bootstrap_seed = 20261106

inventory_path = Path("01_inventory.ipynb")
if not inventory_path.is_file():
    inventory_path = Path("notebooks") / "01_inventory.ipynb"
with capture_output(stdout=False, stderr=False):
    get_ipython().run_line_magic("run", f'"{inventory_path.resolve().as_posix()}"')

# A 3-month label formed after September 2012 is not known by the end of the training window.
train_start, train_end = pd.Timestamp("2000-01-01"), pd.Timestamp("2012-09-30")
test_start, test_end = pd.Timestamp("2013-01-01"), pd.Timestamp("2022-09-30")

## Modelling table

`log_size_rank` is the log of the security's market-cap rank among eligible non-members that month, with 1 as the largest. The rank comes from `mktcap_pct_rank`, which notebook 01 standardizes without winsorizing, so its order is intact. It is needed because winsorizing `log_mktcap` at the 99th percentile gives the largest 1% of non-members the same value, and those are the most likely candidates.

A company can have one share class in the index and another outside it, such as Berkshire Hathaway class A and class B. Notebook 01 treats the second class as a non-member, but it is not a candidate for this strategy. This notebook removes a security when another security of the same CRSP `PERMCO` is a member at that month-end, and ranks size after that step. The removed rows are counted below.

Rows with a missing `y3` are dropped, as notebook 01 requires. Formation dates from October to December 2012 form a gap between training and test, so no training label depends on information from the test period.

In [ ]:
table = features.loc[features["y3"].notna()].copy()
table["y3"] = table["y3"].astype(int)

firm_ids = pd.read_csv(
    paths["crsp_monthly"], usecols=["PERMNO", "PERMCO", "MthCalDt"], parse_dates=["MthCalDt"]
).rename(columns={"PERMNO": "permno", "PERMCO": "permco", "MthCalDt": "mthcaldt"}).drop_duplicates()
if firm_ids.duplicated(["permno", "mthcaldt"]).any():
    raise ValueError("A security has more than one PERMCO in a month.")
member_firms = monthly_panel.loc[monthly_panel["in_sp500"], ["permno", "mthcaldt"]].merge(
    firm_ids, on=["permno", "mthcaldt"], validate="one_to_one"
)[["permco", "mthcaldt"]].drop_duplicates().assign(firm_in_sp500=True)
table = table.merge(firm_ids, on=["permno", "mthcaldt"], how="left", validate="one_to_one").merge(
    member_firms, on=["permco", "mthcaldt"], how="left", validate="many_to_one"
)
if table["permco"].isna().any():
    raise ValueError("A candidate row has no PERMCO.")
other_class_member = table["firm_in_sp500"].eq(True)
display(pd.Series({
    "Rows removed": int(other_class_member.sum()),
    "Positive rows removed": int(table.loc[other_class_member, "y3"].sum()),
    "Securities removed": table.loc[other_class_member, "permno"].nunique(),
    "Rows kept": int((~other_class_member).sum()),
}, name="another share class is a member"))
table = table.loc[~other_class_member].drop(columns="firm_in_sp500")

table["size_rank"] = table.groupby("mthcaldt")["mktcap_pct_rank"].rank(ascending=False, method="min")
table["log_size_rank"] = np.log(table["size_rank"])
table = table.rename(columns={"mthcaldt": "formation_date"})

rank_check = table.groupby("formation_date").apply(
    lambda month: month["size_rank"].corr(month["log_mktcap"], method="spearman"), include_groups=False
)
if rank_check.gt(-0.99).any():
    raise ValueError("The size rank does not match market capitalization.")

# Recompute the months until the next effective date and confirm it reproduces y3.
table = pd.merge_asof(
    table.sort_values("formation_date", kind="stable"), label_targets,
    left_on="formation_date", right_on="effective_date", by="permno",
    direction="forward", allow_exact_matches=False,
).rename(columns={"effective_date": "next_effective"})
table["lead_months"] = (
    (table["next_effective"].dt.year - table["formation_date"].dt.year) * 12
    + table["next_effective"].dt.month - table["formation_date"].dt.month
)
if not table["lead_months"].le(3).astype(int).eq(table["y3"]).all():
    raise ValueError("Lead times do not reproduce the y3 label.")

table["split"] = np.select(
    [table["formation_date"].between(train_start, train_end),
     table["formation_date"].between(test_start, test_end)],
    ["train", "test"], default="unused",
)
if table.loc[table["split"].eq("test"), "formation_date"].max().to_period("M") != test_end.to_period("M"):
    raise ValueError("The test window does not reach its expected final month.")

model_inputs = {
    "S: size rank": ["log_size_rank"],
    "B: size rank + liquidity (baseline)": ["log_size_rank", "log_dollar_volume", "dollar_volume_to_mktcap"],
    "N: notebook 01 size + liquidity": ["log_mktcap", "mktcap_pct_rank", "log_dollar_volume", "dollar_volume_to_mktcap"],
    "C: all features + size rank": ["log_size_rank", *feature_columns],
}
primary_model = "B: size rank + liquidity (baseline)"
all_inputs = sorted({column for inputs in model_inputs.values() for column in inputs})
table["complete_case"] = table[all_inputs].notna().all(axis=1)

used = table.loc[table["split"].isin(["train", "test"])]
display(used.groupby("split").agg(
    first_month=("formation_date", "min"), last_month=("formation_date", "max"),
    months=("formation_date", "nunique"), rows=("y3", "size"), positive_rows=("y3", "sum"),
    base_rate=("y3", "mean"),
).loc[["train", "test"]])

missing = pd.DataFrame({
    (split, kind): {
        column: (rows if kind == "all rows" else rows.loc[rows["y3"].eq(1)])[column].isna().mean()
        for column in all_inputs
    }
    for split, rows in used.groupby("split") for kind in ["all rows", "positive rows"]
}).round(4)
display(missing.rename_axis("share missing"))

positive_ranks = used.loc[used["y3"].eq(1)].groupby("split")["size_rank"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
display(positive_ranks.loc[["train", "test"]].round(1).rename_axis("size rank of positive rows"))

## Check event coverage

An event can be labelled only if its security appears in the candidate table during the three months before the effective date. The table classifies each uncovered addition using its CRSP rows in those months. These additions cannot be predicted by this model.

In [ ]:
panel_months = monthly_panel["mthcaldt"].dt.year * 12 + monthly_panel["mthcaldt"].dt.month
prior_rows = monthly_panel[["permno", "in_sp500", "eligible"]].assign(panel_month=panel_months)


def coverage(start, end):
    window_events = label_targets.loc[label_targets["effective_date"].between(start, end)].copy()
    observed = table.loc[table["y3"].eq(1), ["permno", "next_effective"]].drop_duplicates()
    window_events["covered"] = window_events.merge(
        observed, left_on=["permno", "effective_date"], right_on=["permno", "next_effective"], how="left"
    )["next_effective"].notna().to_numpy()
    uncovered = window_events.loc[~window_events["covered"]].copy()
    uncovered["event_month"] = uncovered["effective_date"].dt.year * 12 + uncovered["effective_date"].dt.month
    rows = uncovered.merge(prior_rows, on="permno")
    rows = rows.loc[(rows["event_month"] - rows["panel_month"]).between(1, 3)]
    status = rows.groupby(["permno", "effective_date"]).agg(
        member=("in_sp500", "any"), eligible=("eligible", "any")
    ).reset_index()
    uncovered = uncovered.merge(status, on=["permno", "effective_date"], how="left")
    reason = np.select(
        [uncovered["member"].isna(), uncovered["member"].eq(True), uncovered["eligible"].eq(True)],
        ["no CRSP month in the prior 3", "already a member", "another share class is a member"],
        default="ineligible security",
    )
    return {
        "additions": len(window_events), "covered": int(window_events["covered"].sum()),
        "coverage_share": round(window_events["covered"].mean(), 3),
        **pd.Series(reason).value_counts().to_dict(),
    }


display(pd.DataFrame({
    "train labels": coverage(train_start + pd.offsets.MonthBegin(1), train_end + pd.offsets.MonthEnd(3)),
    "test labels": coverage(test_start + pd.offsets.MonthBegin(1), test_end + pd.offsets.MonthEnd(3)),
}).T.fillna(0))

## Fit the logits on the training window

Each model uses the rows with all of its inputs. Coefficients stay fixed through the test. Monthly rows for one security overlap, and rows within a month share conditions, so standard errors are clustered by security and by month. Model B was chosen as the baseline before looking at test results. The others show what the size rank and the remaining features add.

In [ ]:
def fit_logit(inputs):
    sample = table.loc[table["split"].eq("train")].dropna(subset=inputs)
    groups = np.column_stack([pd.factorize(sample["permno"])[0], pd.factorize(sample["formation_date"])[0]])
    return sm.Logit(sample["y3"], sm.add_constant(sample[inputs])).fit(
        disp=False, maxiter=100, cov_type="cluster", cov_kwds={"groups": groups}
    )


fits = {name: fit_logit(inputs) for name, inputs in model_inputs.items()}
if not all(fit.mle_retvals["converged"] for fit in fits.values()):
    raise RuntimeError("A logit did not converge.")

display(pd.concat({
    name: pd.DataFrame({"coefficient": fit.params, "clustered_se": fit.bse, "t": fit.tvalues})
    for name, fit in fits.items()
}).round(3))
display(pd.DataFrame({
    name: {"rows": int(fit.nobs), "positive_rows": int(fit.model.endog.sum()), "pseudo_r2": round(fit.prsquared, 4)}
    for name, fit in fits.items()
}).T)

for name, inputs in model_inputs.items():
    scorable = table[inputs].notna().all(axis=1)
    table[name] = np.nan
    table.loc[scorable, name] = fits[name].predict(sm.add_constant(table.loc[scorable, inputs], has_constant="add"))

## Evaluate out of sample

- **Precision at 20** is the share of the monthly top 20 joining within three months. Only a few securities are positive in a typical month, so it cannot reach 100%. *Maximum attainable* gives its ceiling and *base rate* the outcome of random selection.
- **Row recall at 20** is the share of positive rows in the top 20. **Event recall at 20** is the share of additions in the top 20 at least once before they took effect.
- **Top-decile recall** is the share of positive rows in the monthly top 10% of scores.
- **AUC** is 0.5 for a random ranking and 1 for a perfect one. *Common sample* compares all models on the same rows.
- Confidence intervals use a 12-month moving-block bootstrap of the monthly values, because overlapping labels are serially correlated.

In [ ]:
def select_top(rows, score, n=top_n):
    return (
        rows.dropna(subset=[score])
        .sort_values(["formation_date", score, "permno"], ascending=[True, False, True], kind="stable")
        .groupby("formation_date").head(n)
    )


def block_bootstrap_ci(monthly_values):
    values = monthly_values.dropna().to_numpy()
    generator = np.random.default_rng(bootstrap_seed)
    blocks_needed = int(np.ceil(len(values) / bootstrap_block_months))
    starts = generator.integers(0, len(values) - bootstrap_block_months + 1, (bootstrap_draws, blocks_needed))
    offsets = starts[:, :, None] + np.arange(bootstrap_block_months)
    means = values[offsets].reshape(bootstrap_draws, -1)[:, :len(values)].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])


def evaluate(rows, score):
    scored = rows.dropna(subset=[score])
    selected = select_top(scored, score)
    monthly_precision = selected.groupby("formation_date")["y3"].mean()
    monthly_auc = scored.groupby("formation_date").apply(
        lambda month: roc_auc_score(month["y3"], month[score]) if month["y3"].nunique() == 2 else np.nan,
        include_groups=False,
    )
    decile = scored.groupby("formation_date")[score].rank(ascending=False, pct=True).le(0.10)
    positives_by_month = scored.groupby("formation_date")["y3"].sum()
    event_key = ["permno", "next_effective"]
    events_all = scored.loc[scored["y3"].eq(1), event_key].drop_duplicates()
    events_hit = selected.loc[selected["y3"].eq(1), event_key].drop_duplicates()
    precision_ci, auc_ci = block_bootstrap_ci(monthly_precision), block_bootstrap_ci(monthly_auc)
    common = scored.loc[scored["complete_case"]]
    return {
        "rows": len(scored), "positive_rows": int(scored["y3"].sum()),
        "precision_at_20": monthly_precision.mean(),
        "precision_ci_low": precision_ci[0], "precision_ci_high": precision_ci[1],
        "maximum_attainable": positives_by_month.clip(upper=top_n).div(top_n).mean(),
        "base_rate": scored["y3"].mean(),
        "row_recall_at_20": selected["y3"].sum() / scored["y3"].sum(),
        "event_recall_at_20": len(events_hit) / len(events_all),
        "top_decile_recall": scored.loc[decile, "y3"].sum() / scored["y3"].sum(),
        "auc_pooled": roc_auc_score(scored["y3"], scored[score]),
        "auc_common_sample": roc_auc_score(common["y3"], common[score]),
        "auc_monthly_mean": monthly_auc.mean(),
        "auc_monthly_ci_low": auc_ci[0], "auc_monthly_ci_high": auc_ci[1],
    }


test_rows = table.loc[table["split"].eq("test")]
periods = {
    "2013–09/2022": test_rows,
    "2013–2017": test_rows.loc[test_rows["formation_date"].dt.year.le(2017)],
    "2018–09/2022": test_rows.loc[test_rows["formation_date"].dt.year.ge(2018)],
}
results = pd.DataFrame({
    (period, name): evaluate(rows, name) for period, rows in periods.items() for name in model_inputs
}).T.rename_axis(["test period", "model"])
display(results.round(4).T)

in_sample = pd.DataFrame({
    name: evaluate(table.loc[table["split"].eq("train")], name) for name in model_inputs
}).T[["precision_at_20", "maximum_attainable", "row_recall_at_20", "auc_pooled"]]
display(in_sample.round(4).rename_axis("training window (in sample)"))

## Are the differences between models reliable?

Separate confidence intervals overlap even when one model is consistently better. This table compares monthly top-20 precision in the same months, against the baseline, using the same block bootstrap. An interval that excludes zero indicates a difference, though these comparisons were not all planned in advance.

In [ ]:
monthly_precision = {
    name: select_top(test_rows, name).groupby("formation_date")["y3"].mean() for name in model_inputs
}
comparisons = {}
for period, rows in periods.items():
    months = rows["formation_date"].unique()
    for name in model_inputs:
        if name == primary_model:
            continue
        difference = (monthly_precision[name] - monthly_precision[primary_model]).loc[months]
        low, high = block_bootstrap_ci(difference)
        comparisons[(period, name)] = {"difference_vs_baseline": difference.mean(), "ci_low": low, "ci_high": high}
display(pd.DataFrame(comparisons).T.rename_axis(["test period", "model"]).round(4))

## Lead time and calibration

`lead_months` is the number of calendar months between formation and the effective date. S&P usually announces an addition shortly before it takes effect, so a selection with a lead of one month may already be public. The second table therefore recomputes the baseline's precision with lead-one positives removed from the numerator.

Calibration compares predicted and realized rates. The fixed coefficients do not adapt if additions become more or less frequent.

In [ ]:
primary_selected = select_top(test_rows, primary_model)
display(pd.DataFrame({
    "all positive rows": test_rows.loc[test_rows["y3"].eq(1), "lead_months"].value_counts().sort_index(),
    "in the top 20": primary_selected.loc[primary_selected["y3"].eq(1), "lead_months"].value_counts().sort_index(),
}).rename_axis("lead_months"))

lead_two_plus = primary_selected["y3"].eq(1) & primary_selected["lead_months"].ge(2)
lead_series = lead_two_plus.groupby(primary_selected["formation_date"]).mean()
lead_ci = block_bootstrap_ci(lead_series)
display(pd.Series({
    "Precision at 20, all leads": primary_selected["y3"].mean(),
    "Precision at 20, leads of 2–3 months": lead_series.mean(),
    "95% interval, low": lead_ci[0],
    "95% interval, high": lead_ci[1],
}, name="baseline").round(4))

scored_test = test_rows.dropna(subset=[primary_model]).copy()
score_rank = scored_test.groupby("formation_date")[primary_model].rank(ascending=False, method="first")
month_size = scored_test.groupby("formation_date")[primary_model].transform("size")
scored_test["score_bucket"] = pd.cut(
    (score_rank - 1) / month_size, bins=[0, 0.01, 0.05, 0.10, 0.25, 0.50, 1.0], include_lowest=True, right=False,
    labels=["top 1%", "1–5%", "5–10%", "10–25%", "25–50%", "bottom 50%"],
)
calibration = scored_test.groupby("score_bucket", observed=True).agg(
    rows=("y3", "size"), positive_rows=("y3", "sum"),
    mean_predicted=(primary_model, "mean"), realized=("y3", "mean"),
)
calibration.loc["top 20"] = [
    len(primary_selected), primary_selected["y3"].sum(),
    primary_selected[primary_model].mean(), primary_selected["y3"].mean(),
]
calibration.loc["all"] = [
    len(scored_test), scored_test["y3"].sum(), scored_test[primary_model].mean(), scored_test["y3"].mean(),
]
display(calibration.round(5))

## Dated candidate list

`anticipation_baseline_selections.csv` is written to the Git-ignored `Data/prepared/` directory. It lists the baseline's top 20 at each test formation date and holds no outcomes. Join labels and returns by `permno` and `formation_date` in a later step. The final table shows the last list with its outcomes as a quick check. Clear this output before sharing the notebook, since it contains WRDS-derived data.

In [ ]:
selection_names = pd.merge_asof(
    primary_selected.sort_values("formation_date", kind="stable"),
    names[["permno", "ticker", "secinfostartdt"]].sort_values("secinfostartdt"),
    left_on="formation_date", right_on="secinfostartdt", by="permno", direction="backward",
)
selection_names["list_rank"] = selection_names.groupby("formation_date")[primary_model].rank(
    ascending=False, method="first"
).astype(int)
selections = selection_names.rename(columns={primary_model: "predicted_probability"})[
    ["formation_date", "list_rank", "permno", "ticker", "size_rank", "predicted_probability"]
].sort_values(["formation_date", "list_rank"], ignore_index=True)
if selections.duplicated(["formation_date", "permno"]).any():
    raise ValueError("A security appears twice in one candidate list.")
if selections.groupby("formation_date").size().ne(top_n).any():
    raise ValueError("A candidate list does not contain the expected number of securities.")

if save_prepared:
    prepared_dir = data_dir / "prepared"
    prepared_dir.mkdir(parents=True, exist_ok=True)
    selections.to_csv(prepared_dir / "anticipation_baseline_selections.csv", index=False)

last_list = selections.loc[selections["formation_date"].eq(selections["formation_date"].max())]
display(last_list.merge(
    table[["permno", "formation_date", "y3", "next_effective"]], on=["permno", "formation_date"], validate="one_to_one"
).round(4))

## Add the S&P earnings criterion

S&P's rule requires positive earnings in the latest quarter and over the latest four quarters combined. This section tests that criterion, using `ibq`, income before extraordinary items, as the closest available field to GAAP earnings excluding discontinued operations.

**No look-ahead.** The local Compustat export has no report date (`RDQ`), so a quarter counts as known only four months after its end, as the project plan specifies. A three-month lag is shown as a sensitivity. The criterion is unknown if the security has no valid CRSP–Compustat link, the four quarters are not consecutive, or the newest usable quarter ended more than 12 months before formation. Unknown stays a separate category rather than a pass or a fail.

**Limits.** The export contains current Compustat values rather than as-first-reported vintages, so restatements can affect the signal. The current rule is applied to the whole sample, and S&P's historical wording may have differed.

Model E, chosen before looking at its test results, is the baseline plus `earnings_pass` and `earnings_unknown`. List H keeps the baseline's score but selects only from securities that pass. Model F adds the same two variables to model C.

In [ ]:
quarterly = pd.read_csv(
    paths["comp_quarterly"], usecols=["gvkey", "datadate", "ibq"], parse_dates=["datadate"]
).dropna(subset=["ibq"])
# A fiscal-year change can report one quarter twice with the same values.
quarterly = quarterly.drop_duplicates(["gvkey", "datadate", "ibq"])
quarterly = quarterly.sort_values(["gvkey", "datadate"], ignore_index=True)
if quarterly.duplicated(["gvkey", "datadate"]).any():
    conflicts = quarterly.duplicated(["gvkey", "datadate"], keep=False)
    print(f"Quarters with conflicting duplicates removed: {conflicts.sum():,} rows")
    quarterly = quarterly.loc[~conflicts].reset_index(drop=True)
quarterly["quarter_month"] = quarterly["datadate"].dt.year * 12 + quarterly["datadate"].dt.month
by_firm = quarterly.groupby("gvkey")
quarterly["earnings_4q"] = by_firm["ibq"].rolling(4, min_periods=4).sum().droplevel(0)
quarterly["consecutive_4q"] = quarterly["quarter_month"] - by_firm["quarter_month"].shift(3) == 9
quarterly["passes_rule"] = quarterly["ibq"].gt(0) & quarterly["earnings_4q"].gt(0)

links = pd.read_csv(paths["ccm_link"], usecols=["gvkey", "LPERMNO", "LINKPRIM", "LINKDT", "LINKENDDT"])
links = links.loc[links["LINKPRIM"].isin(["P", "C"])].rename(columns={"LPERMNO": "permno"}).copy()
links["link_start"] = pd.to_datetime(links["LINKDT"])
links["link_end"] = pd.to_datetime(links["LINKENDDT"].replace("E", "2099-12-31"))

rows_for_link = table[["permno", "formation_date"]].drop_duplicates()
linked = rows_for_link.merge(links[["permno", "gvkey", "link_start", "link_end"]], on="permno")
linked = linked.loc[linked["formation_date"].between(linked["link_start"], linked["link_end"]), ["permno", "formation_date", "gvkey"]]
ambiguous_links = linked.duplicated(["permno", "formation_date"], keep=False)
print(f"Security-months with more than one valid link, treated as unknown: {ambiguous_links.sum():,}")
linked = linked.loc[~ambiguous_links]


def earnings_status(lag_months, max_age_months=12):
    frame = linked.assign(
        formation_month=linked["formation_date"].dt.year * 12 + linked["formation_date"].dt.month
    ).sort_values("formation_month", kind="stable")
    known = quarterly.assign(known_month=quarterly["quarter_month"] + lag_months).sort_values("known_month", kind="stable")
    matched = pd.merge_asof(
        frame, known[["gvkey", "known_month", "quarter_month", "consecutive_4q", "passes_rule"]],
        left_on="formation_month", right_on="known_month", by="gvkey", direction="backward",
    )
    if (matched["formation_month"] - matched["quarter_month"]).lt(lag_months).any():
        raise ValueError("An earnings quarter was used before its assumed availability date.")
    usable = matched["consecutive_4q"].eq(True) & (matched["formation_month"] - matched["quarter_month"]).le(max_age_months)
    matched["status"] = np.where(usable, np.where(matched["passes_rule"].eq(True), "pass", "fail"), "unknown")
    return table[["permno", "formation_date"]].merge(
        matched[["permno", "formation_date", "status"]], on=["permno", "formation_date"], how="left", validate="one_to_one"
    )["status"].fillna("unknown").to_numpy()


table["earnings_status"] = earnings_status(4)
table["earnings_status_lag3"] = earnings_status(3)
for suffix, column in [("", "earnings_status"), ("_lag3", "earnings_status_lag3")]:
    table[f"earnings_pass{suffix}"] = table[column].eq("pass").astype(int)
    table[f"earnings_unknown{suffix}"] = table[column].eq("unknown").astype(int)

training = table.loc[table["split"].eq("train")]
top_100_train = training.loc[training["size_rank"].le(100)]
display(pd.DataFrame({
    "all training rows": training["earnings_status"].value_counts(normalize=True),
    "size rank 1–100": top_100_train["earnings_status"].value_counts(normalize=True),
    "positive rows": training.loc[training["y3"].eq(1), "earnings_status"].value_counts(normalize=True),
    "addition rate, size rank 1–100": top_100_train.groupby("earnings_status")["y3"].mean(),
}).round(4).rename_axis("earnings status, training window"))

In [ ]:
earnings_inputs = ["earnings_pass", "earnings_unknown"]
extended_inputs = {
    "E: baseline + earnings": [*model_inputs[primary_model], *earnings_inputs],
    "E3: same, 3-month lag": [*model_inputs[primary_model], "earnings_pass_lag3", "earnings_unknown_lag3"],
    "F: all features + earnings": [*model_inputs["C: all features + size rank"], *earnings_inputs],
}
extended_fits = {name: fit_logit(inputs) for name, inputs in extended_inputs.items()}
if not all(fit.mle_retvals["converged"] for fit in extended_fits.values()):
    raise RuntimeError("A logit did not converge.")
for name, inputs in extended_inputs.items():
    scorable = table[inputs].notna().all(axis=1)
    table[name] = np.nan
    table.loc[scorable, name] = extended_fits[name].predict(
        sm.add_constant(table.loc[scorable, inputs], has_constant="add")
    )
hard_filter = "H: baseline, passing securities only"
table[hard_filter] = table[primary_model].where(table["earnings_status"].eq("pass"))

fit_e = extended_fits["E: baseline + earnings"]
display(pd.DataFrame({"coefficient": fit_e.params, "clustered_se": fit_e.bse, "t": fit_e.tvalues}).round(3).rename_axis("model E"))


def list_metrics(rows, score, reference=None):
    selected = select_top(rows, score)
    monthly = selected.groupby("formation_date")["y3"].mean().reindex(rows["formation_date"].unique())
    low, high = block_bootstrap_ci(monthly)
    event_key = ["permno", "next_effective"]
    events_all = rows.loc[rows["y3"].eq(1), event_key].drop_duplicates()
    output = {
        "precision_at_20": monthly.mean(), "precision_ci_low": low, "precision_ci_high": high,
        "row_recall": selected["y3"].sum() / rows["y3"].sum(),
        "event_recall": len(selected.loc[selected["y3"].eq(1), event_key].drop_duplicates()) / len(events_all),
        "leads_2_3_precision": (selected["y3"].eq(1) & selected["lead_months"].ge(2)).groupby(selected["formation_date"]).mean().mean(),
    }
    if reference is not None:
        difference = monthly - select_top(rows, reference).groupby("formation_date")["y3"].mean()
        output["difference_vs_baseline"] = difference.mean()
        output["difference_ci_low"], output["difference_ci_high"] = block_bootstrap_ci(difference)
    return output


test_rows = table.loc[table["split"].eq("test")]
periods = {
    "2013–09/2022": test_rows,
    "2013–2017": test_rows.loc[test_rows["formation_date"].dt.year.le(2017)],
    "2018–09/2022": test_rows.loc[test_rows["formation_date"].dt.year.ge(2018)],
}
comparison_models = [primary_model, *extended_inputs, hard_filter]
earnings_results = pd.DataFrame({
    (period, name): list_metrics(rows, name, None if name == primary_model else primary_model)
    for period, rows in periods.items() for name in comparison_models
}).T.rename_axis(["test period", "model"])
display(earnings_results.round(4))
display(pd.Series({
    name: roc_auc_score(test_rows.dropna(subset=[name])["y3"], test_rows.dropna(subset=[name])[name])
    for name in [primary_model, *extended_inputs]
}, name="pooled AUC, 2013–09/2022").round(4))

model_e = "E: baseline + earnings"
selected_e = select_top(test_rows, model_e)
display(selected_e.loc[selected_e["formation_date"].eq(selected_e["formation_date"].max()),
    ["permno", "size_rank", "earnings_status", model_e, "y3", "next_effective"]].round(4))
display(pd.DataFrame({
    "baseline": select_top(test_rows, primary_model)["earnings_status"].value_counts(normalize=True),
    "model E": selected_e["earnings_status"].value_counts(normalize=True),
}).round(3).rename_axis("earnings status in the top 20"))

## Ex-ante portfolio

The earlier sections measure prediction. This section measures returns, using the plan's test for selection bias: the portfolio holds every selected security, including those that never join the index.

**Construction.** At each month-end `t`, select the list using data available at `t`, weight its securities equally, and hold them through month `t+1`. Formation dates run from January 2013 to September 2022, so returns cover February 2013 to October 2022. As in notebook 02, the return is CRSP `MthRet`, delisting flags `G`, `M`, and `V` count as missing, and a month with a missing holding return stays blank.

**Rules chosen before seeing the results.** Model E is the main portfolio. The baseline, size-only list, and the 100 largest non-members are comparisons. The last one shows whether a list adds anything beyond holding large non-members. Alpha uses the five Fama–French factors and momentum with 12 Newey–West lags, as in notebook 02.

**What this does not measure.** Returns are gross. Selection assumes trading at the same month-end close used for the signal. A security that joins is held to the end of the month instead of being sold at the effective-date close, which is leg 2's task. Some selections may already be announced. The break-even is the round-trip cost per dollar traded at which alpha reaches zero, and it uses list changes only, ignoring rebalancing to equal weights. The sample without 2020 treats December 2019 and January 2021 as adjacent months when computing Newey–West lags.

In [ ]:
five_factors = read_monthly_factors(
    data_dir / "F-F_Research_Data_5_Factors_2x3_CSV.zip", ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
)
momentum_factor = read_monthly_factors(data_dir / "F-F_Momentum_Factor_CSV.zip", ["Mom"])
monthly_factors = five_factors.join(momentum_factor, validate="one_to_one")
factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]

market_returns = pd.read_csv(paths["index_monthly"], usecols=["mthcaldt", "vwretd"], parse_dates=["mthcaldt"])
market_returns = market_returns.assign(month=market_returns["mthcaldt"].dt.to_period("M")).set_index("month")["vwretd"]

stock_returns = monthly_panel[["permno", "mthcaldt", "mthret", "mthdelflg"]].copy()
stock_returns["return_month"] = stock_returns.pop("mthcaldt").dt.to_period("M")
stock_returns["stock_return"] = stock_returns.pop("mthret").mask(stock_returns.pop("mthdelflg").isin(["G", "M", "V"]))
if stock_returns.duplicated(["permno", "return_month"]).any():
    raise ValueError("A security has more than one return in a month.")

table["largest_100"] = -table["size_rank"]
test_rows = table.loc[table["split"].eq("test")]
portfolio_lists = {
    "E: baseline + earnings (main)": select_top(test_rows, "E: baseline + earnings"),
    "B: baseline": select_top(test_rows, primary_model),
    "S: size rank": select_top(test_rows, "S: size rank"),
    "Largest 100 non-members": select_top(test_rows, "largest_100", n=100),
}


def holding_returns(selected):
    holdings = selected[["permno", "formation_date", "y3", "lead_months"]].copy()
    holdings["return_month"] = holdings["formation_date"].dt.to_period("M") + 1
    return holdings.merge(stock_returns, on=["permno", "return_month"], how="left", validate="one_to_one")


def monthly_series(selected):
    holdings = holding_returns(selected)
    series = holdings.groupby("return_month").agg(
        positions=("permno", "size"), returns=("stock_return", "count"), portfolio=("stock_return", "mean")
    )
    series["portfolio"] = series["portfolio"].where(series["positions"].eq(series["returns"]))
    members = selected.groupby("formation_date")["permno"].agg(set)
    replaced = [len(now - before) / len(now) for before, now in zip(members.iloc[:-1], members.iloc[1:])]
    series["replaced_share"] = np.r_[np.nan, replaced]
    series = series.join(market_returns.rename("market")).join(monthly_factors)
    series["excess"] = series["portfolio"] - series["RF"]
    series["market_adjusted"] = series["portfolio"] - series["market"]
    return series


portfolio_series = {name: monthly_series(selected) for name, selected in portfolio_lists.items()}
for name, series in portfolio_series.items():
    if series[factor_columns].isna().any().any():
        raise ValueError(f"Factors do not cover every return month for {name}.")

portfolio_samples = {
    "02/2013–10/2022": lambda index: index.year >= 0,
    "02/2013–2017": lambda index: index.year <= 2017,
    "2018–10/2022": lambda index: index.year >= 2018,
    "Without 2020": lambda index: index.year != 2020,
}


def portfolio_summary(series):
    usable = series.dropna(subset=["portfolio"])
    # Newey-West lags follow row order, so the sample without 2020 treats December 2019 and January 2021 as adjacent.
    if len(usable) != len(series):
        print(f"Blank months dropped from the regression: {len(series) - len(usable)}")
    fit = sm.OLS(usable["excess"], sm.add_constant(usable[factor_columns])).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
    adjusted = sm.OLS(usable["market_adjusted"], np.ones(len(usable))).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
    cumulative = (1 + usable["portfolio"]).cumprod()
    relative = (1 + usable["market_adjusted"]).cumprod()
    annual_alpha = fit.params["const"] * 12
    annual_turnover = series["replaced_share"].mean() * 12
    return {
        "months": len(usable),
        "annual_return": cumulative.iloc[-1] ** (12 / len(usable)) - 1,
        "annual_market_return": (1 + usable["market"]).prod() ** (12 / len(usable)) - 1,
        "annual_volatility": usable["portfolio"].std() * np.sqrt(12),
        "sharpe": usable["excess"].mean() / usable["excess"].std() * np.sqrt(12),
        "market_adjusted_annual": adjusted.params.iloc[0] * 12,
        "market_adjusted_t": adjusted.tvalues.iloc[0],
        "alpha_annual": annual_alpha,
        "alpha_t": fit.tvalues["const"],
        "market_beta": fit.params["Mkt-RF"],
        "information_ratio": annual_alpha / (fit.resid.std() * np.sqrt(12)),
        "max_drawdown": (cumulative / cumulative.cummax() - 1).min(),
        "worst_month": usable["portfolio"].min(),
        "annual_one_way_turnover": annual_turnover,
        "break_even_bps": 1e4 * annual_alpha / (2 * annual_turnover),
    }


display(pd.DataFrame({
    name: {"blank months": int(series["portfolio"].isna().sum()), "return months": len(series),
           "missing stock returns": int((series["positions"] - series["returns"]).sum())}
    for name, series in portfolio_series.items()
}).T)
portfolio_results = pd.DataFrame({
    (sample, name): portfolio_summary(series.loc[keep(series.index)])
    for sample, keep in portfolio_samples.items() for name, series in portfolio_series.items()
}).T.rename_axis(["sample", "portfolio"])
display(portfolio_results.round(3))

main_series = portfolio_series["E: baseline + earnings (main)"].dropna(subset=["portfolio"])
main_fit = sm.OLS(main_series["excess"], sm.add_constant(main_series[factor_columns])).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
display(pd.DataFrame({"coefficient": main_fit.params, "t": main_fit.tvalues}).round(3).rename_axis("main portfolio"))

## Who earns the return?

This table splits the main portfolio's holdings using outcomes known only afterward. It describes the source of the return and cannot be traded. If the return comes only from securities that join, the portfolio depends on prediction accuracy. If the non-joiners earn the same return, the result may reflect another characteristic of these securities.

In [ ]:
main_holdings = holding_returns(portfolio_lists["E: baseline + earnings (main)"]).join(market_returns.rename("market"), on="return_month")
main_holdings["market_adjusted"] = main_holdings["stock_return"] - main_holdings["market"]
main_holdings["outcome"] = np.where(
    main_holdings["y3"].eq(0), "does not join within 3 months",
    "joins in " + main_holdings["lead_months"].astype("Int64").astype(str) + " month(s)",
)
display(main_holdings.groupby("outcome").agg(
    holdings=("permno", "size"), securities=("permno", "nunique"),
    mean_market_adjusted=("market_adjusted", "mean"), median_market_adjusted=("market_adjusted", "median"),
).round(4))

earnings_selections = portfolio_lists["E: baseline + earnings (main)"].sort_values(
    ["formation_date", "E: baseline + earnings", "permno"], ascending=[True, False, True]
)[["formation_date", "permno", "size_rank", "earnings_status", "E: baseline + earnings"]].rename(
    columns={"E: baseline + earnings": "predicted_probability"}
)
if save_prepared:
    earnings_selections.to_csv(prepared_dir / "anticipation_earnings_selections.csv", index=False)
print(f"Rows in the main candidate list: {len(earnings_selections):,}")